# Notebook 02 — Experiment Tracking

**RHOAI 3.5 | MLflow Tracking API**

This notebook demonstrates core MLflow experiment tracking on a real dataset:

1. Manual parameter and metric logging
2. Logging artifacts (plots, feature importance)
3. **Autologging** — one line enables automatic capture of params, metrics, and model
4. Model signature and input examples
5. Logging with the `MlflowClient` for programmatic control

---

We train three classifiers on the Wine dataset and track everything in the `rhoai-mlflow-demo` experiment.

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

EXPERIMENT_NAME = "rhoai-mlflow-demo"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Active experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI     : {mlflow.get_tracking_uri()}")

## Prepare the Dataset

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Dataset    : Wine Classification ({len(wine.target_names)} classes)")
print(f"Features   : {X.shape[1]}")
print(f"Train rows : {len(X_train)}")
print(f"Test rows  : {len(X_test)}")
print(f"Classes    : {list(wine.target_names)}")

## Section 1 — Manual Logging

Explicit logging gives you full control: log exactly the params, metrics, and artifacts that matter.

Key API calls:
- `mlflow.log_param(key, value)` — hyperparameters, configuration
- `mlflow.log_params({...})` — batch of params
- `mlflow.log_metric(key, value, step=N)` — scalar metrics, optionally over steps
- `mlflow.log_metrics({...})` — batch of metrics
- `mlflow.set_tag(key, value)` — metadata labels
- `mlflow.log_artifact(path)` — any file (plot, CSV, config)

In [ ]:
import tempfile, os
from mlflow.models import infer_signature
from mlflow.sklearn import SERIALIZATION_FORMAT_CLOUDPICKLE

def train_and_log_rf(n_estimators=100, max_depth=5, run_name="random-forest"):
    with mlflow.start_run(run_name=run_name) as run:

        # --- Parameters ---
        mlflow.log_params({
            "model_type"   : "RandomForestClassifier",
            "n_estimators" : n_estimators,
            "max_depth"    : max_depth,
            "dataset"      : "wine",
            "test_size"    : 0.2,
            "random_state" : 42,
            "scaled"       : True,
        })

        # --- Train ---
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(X_train_scaled, y_train)

        # --- Metrics ---
        train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
        test_acc  = accuracy_score(y_test,  model.predict(X_test_scaled))
        f1        = f1_score(y_test, model.predict(X_test_scaled), average="weighted")

        mlflow.log_metrics({
            "train_accuracy" : train_acc,
            "test_accuracy"  : test_acc,
            "f1_weighted"    : f1,
        })

        # --- Tags ---
        mlflow.set_tag("notebook",    "02_experiment_tracking")
        mlflow.set_tag("team",        "data-science")
        mlflow.set_tag("rhoai_build", "3.5")

        # --- Artifact: Feature Importance Plot ---
        importances = pd.Series(model.feature_importances_, index=wine.feature_names)
        importances = importances.sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(10, 5))
        importances.plot.bar(ax=ax, color="steelblue")
        ax.set_title(f"Feature Importance — RandomForest (n={n_estimators}, depth={max_depth})")
        ax.set_ylabel("Importance Score")
        plt.tight_layout()

        with tempfile.TemporaryDirectory() as tmp:
            plot_path = os.path.join(tmp, "feature_importance.png")
            fig.savefig(plot_path)
            mlflow.log_artifact(plot_path, artifact_path="plots")
        plt.close(fig)

        # --- Artifact: Classification Report ---
        report = classification_report(
            y_test, model.predict(X_test_scaled),
            target_names=wine.target_names, output_dict=False
        )
        with tempfile.TemporaryDirectory() as tmp:
            rpt_path = os.path.join(tmp, "classification_report.txt")
            with open(rpt_path, "w") as f:
                f.write(report)
            mlflow.log_artifact(rpt_path, artifact_path="reports")

        # --- Log Model with signature ---
        # serialization_format=cloudpickle avoids skops trusted-type prompts (MLflow 3.13 default is skops)
        signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
        input_example = X_test_scaled[:3]

        mlflow.sklearn.log_model(
            model,
            name="model",
            serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE,
            signature=signature,
            input_example=input_example,
            registered_model_name=None,
        )

        print(f"Run: {run_name}")
        print(f"  Run ID       : {run.info.run_id}")
        print(f"  Train acc    : {train_acc:.4f}")
        print(f"  Test acc     : {test_acc:.4f}")
        print(f"  F1 weighted  : {f1:.4f}")
        return run.info.run_id


rf_run_id = train_and_log_rf(n_estimators=100, max_depth=5)

## Section 2 — Autologging

`mlflow.sklearn.autolog()` automatically captures:
- All `fit()` parameters
- Training/test metrics
- The fitted model
- A confusion matrix artifact
- Feature importance (for tree models)

One line replaces dozens of `log_param` / `log_metric` calls.

In [ ]:
# Enable autologging for scikit-learn
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True,
    log_post_training_metrics=True,
    silent=False,
)

# Train a GradientBoostingClassifier — autolog captures everything
with mlflow.start_run(run_name="gradient-boosting-autolog") as run:
    mlflow.set_tags({
        "notebook"    : "02_experiment_tracking",
        "autolog"     : "true",
        "rhoai_build" : "3.5",
    })

    gb = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=42,
    )
    gb.fit(X_train_scaled, y_train)

    # Post-training evaluation (autolog will capture these too)
    test_preds = gb.predict(X_test_scaled)
    print(f"GradientBoosting accuracy: {accuracy_score(y_test, test_preds):.4f}")
    print(f"Run ID: {run.info.run_id}")

    gb_run_id = run.info.run_id

# Disable autolog after the run to avoid interfering with manual runs below
mlflow.sklearn.autolog(disable=True)
print("Autologging disabled.")

## Section 3 — Logistic Regression with Step-Based Metrics

Log metrics per regularization strength (treated as "step") to produce a chart in the dashboard.

In [ ]:
C_values = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]

with mlflow.start_run(run_name="logistic-regression-C-sweep") as run:
    mlflow.log_params({
        "model_type"  : "LogisticRegression",
        "solver"      : "lbfgs",
        "max_iter"    : 1000,
        "C_values"    : str(C_values),
        "dataset"     : "wine",
    })
    mlflow.set_tags({
        "notebook"    : "02_experiment_tracking",
        "rhoai_build" : "3.5",
    })

    best_acc, best_C = 0, None

    for step, C in enumerate(C_values):
        lr = LogisticRegression(C=C, solver="lbfgs", max_iter=1000, random_state=42)
        lr.fit(X_train_scaled, y_train)

        train_acc = accuracy_score(y_train, lr.predict(X_train_scaled))
        test_acc  = accuracy_score(y_test,  lr.predict(X_test_scaled))

        mlflow.log_metrics({"C_value": C, "train_accuracy": train_acc, "test_accuracy": test_acc}, step=step)

        if test_acc > best_acc:
            best_acc, best_C, best_model = test_acc, C, lr

    mlflow.log_param("best_C", best_C)
    mlflow.log_metric("best_test_accuracy", best_acc)

    signature = infer_signature(X_train_scaled, best_model.predict(X_train_scaled))
    mlflow.sklearn.log_model(
        best_model,
        name="model",
        serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE,
        signature=signature,
    )

    lr_run_id = run.info.run_id
    print(f"Best C={best_C} → test accuracy={best_acc:.4f}")
    print(f"Run ID: {lr_run_id}")

## Section 4 — Query and Review Runs Programmatically

Use `mlflow.search_runs()` to retrieve and compare run results without leaving the notebook.

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.notebook = '02_experiment_tracking'",
    order_by=["metrics.test_accuracy DESC"],
)

cols = ["run_id", "tags.mlflow.runName", "params.model_type",
        "metrics.train_accuracy", "metrics.test_accuracy", "metrics.f1_weighted"]

available_cols = [c for c in cols if c in runs_df.columns]
print(f"Found {len(runs_df)} runs from this notebook:\n")
print(runs_df[available_cols].to_string(index=False))

In [ ]:
# Visual comparison of test accuracy across all runs
if "metrics.test_accuracy" in runs_df.columns and "tags.mlflow.runName" in runs_df.columns:
    plot_df = runs_df[["tags.mlflow.runName", "metrics.test_accuracy"]].dropna()
    plot_df = plot_df.sort_values("metrics.test_accuracy", ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(plot_df["tags.mlflow.runName"], plot_df["metrics.test_accuracy"],
                   color=["#2ca02c" if v == plot_df["metrics.test_accuracy"].max() else "#1f77b4"
                          for v in plot_df["metrics.test_accuracy"]])
    ax.set_xlabel("Test Accuracy")
    ax.set_title("Run Comparison — Test Accuracy")
    ax.set_xlim(0.8, 1.01)
    for bar, val in zip(bars, plot_df["metrics.test_accuracy"]):
        ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data for comparison chart yet.")

## Section 5 — System Metrics

MLflow 3.x can log system metrics (CPU, memory) automatically. The RHOAI Dashboard shows these in a separate tab alongside your training metrics.

In [ ]:
import time

# Enable system metrics logging (CPU, memory, GPU if available)
mlflow.enable_system_metrics_logging()

with mlflow.start_run(run_name="rf-with-system-metrics") as run:
    mlflow.set_tags({"notebook": "02_experiment_tracking", "system_metrics": "true"})
    mlflow.log_params({
        "model_type"   : "RandomForestClassifier",
        "n_estimators" : 300,
        "max_depth"    : 10,
    })

    model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train)

    time.sleep(1)  # Give system metrics sampler time to collect at least one reading

    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    mlflow.log_metric("test_accuracy", test_acc)
    print(f"Test accuracy: {test_acc:.4f}  | Run ID: {run.info.run_id}")

mlflow.disable_system_metrics_logging()
print("\nCheck the RHOAI Dashboard > Experiments > run > System metrics tab for CPU/memory charts.")

## Summary

| Feature demonstrated | API used |
|---------------------|----------|
| Log params | `mlflow.log_param()`, `log_params()` |
| Log metrics (scalar + step) | `mlflow.log_metric()`, `log_metrics()` |
| Log artifacts (plots, reports) | `mlflow.log_artifact()` |
| Log model with signature | `mlflow.sklearn.log_model()` |
| Autologging | `mlflow.sklearn.autolog()` |
| Query runs | `mlflow.search_runs()` |
| System metrics | `mlflow.enable_system_metrics_logging()` |

**Dashboard:** `Develop & train > Experiments (MLflow) > rhoai-mlflow-demo`

**Next:** `03_hyperparameter_tuning.ipynb` — nested runs and systematic hyperparameter search.